# Ejercicio 2: comparar con evidencia y elegir

El mismo `viajes_features.csv` y la misma preparación importada. Mismo esqueleto del Ejercicio 1, con los cinco modelos de la Semana 10 en un experimento propio. Al final eliges el mejor desde el registro, no desde tu memoria, y compruebas que el modelo guardado reproduce su número.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**). Si vuelves a correr las celdas que entrenan, aparecen corridas repetidas en el registro: es lo esperado.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from preparacion_taxis import dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma (la
# descripcion de sus columnas de entrada). La firma importa al servir el
# modelo, en la Semana 12; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella("viajes_features.csv")
EXPERIMENTO = "tarifa-taxis-comparacion"

Entrenamiento: (2380, 10) Prueba: (595, 10)


## Paso 1: el experimento

In [2]:
# El registro vive en el servidor de MLflow del Codespace (puerto 5050). La
# direccion viene de la variable de entorno MLFLOW_TRACKING_URI, como en un
# equipo de datos: el codigo no la escribe. Sin esa variable (en tu laptop),
# MLflow guarda el registro en una carpeta mlruns al lado del notebook.
print("Registro de MLflow:", mlflow.get_tracking_uri())

# TODO 1: usa el experimento EXPERIMENTO.
mlflow.set_experiment(EXPERIMENTO)

Registro de MLflow: sqlite:////home/satt/icebka/usfq/dmining/ciencia-de-datos/laboratorios/semana-08/data-science-end-to-end/laboratorios/semana-11-sesion-01/desarrollo/mlflow.db


<Experiment: artifact_location='/home/satt/icebka/usfq/dmining/ciencia-de-datos/laboratorios/semana-08/data-science-end-to-end/laboratorios/semana-11-sesion-01/desarrollo/mlruns/2', creation_time=1794117175483, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1794117175483, lifecycle_stage='active', name='tarifa-taxis-comparacion', tags={}, trace_location=None, workspace='default'>

## Paso 2: entrenar y registrar

In [3]:
def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])

    # TODO 2: abre la corrida con el nombre del modelo y registra sus
    # hiperparametros, la semilla de la division (42), la huella de los
    # datos y las filas de entrenamiento.
    mlflow.start_run(run_name=nombre)

    mlflow.log_params({
        **modelo.get_params(),
        "semilla_division": 42,
        "huella_datos": huella_datos,
        "filas_entrenamiento": len(X_ent),
    })

    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))

    # TODO 3: registra las metricas mae_entrenamiento, mae_prueba y
    # segundos, y el pipeline entrenado como el artefacto "modelo".
    mlflow.log_metrics({
        "mae_entrenamiento": mae_ent,
        "mae_prueba": mae_prueba,
        "segundos": segundos,
    })

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="modelo",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree",
            "sklearn.neural_network._stochastic_optimizers.AdamOptimizer",
        ],
    )
    mlflow.end_run()
    print(f"{nombre:20s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}")

## Paso 3: los cinco modelos de la Semana 10

In [4]:
# TODO 4: registra los cinco modelos de la Semana 10, con estos nombres:
# "linea base" (DummyRegressor con la mediana), "ridge" (Ridge con
# alpha=10.0), "bosque aleatorio" (200 arboles, random_state=42),
# "gradient boosting" (random_state=42) y "red neuronal" (MLPRegressor con
# una capa de 64 neuronas, max_iter=1000, random_state=42).
evaluar("linea base", DummyRegressor(strategy="median"))

evaluar("ridge", Ridge(alpha=10.0))

evaluar(
    "bosque aleatorio",
    RandomForestRegressor(n_estimators=200, random_state=42)
)

evaluar(
    "gradient boosting",
    GradientBoostingRegressor(random_state=42)
)

evaluar(
    "red neuronal",
    MLPRegressor(
        hidden_layer_sizes=(64,),
        max_iter=1000,
        random_state=42
    )
)

linea base           entrenamiento 13.1998   prueba 12.5308
ridge                entrenamiento  2.4939   prueba  2.5565
bosque aleatorio     entrenamiento  0.4744   prueba  1.2993
gradient boosting    entrenamiento  1.0705   prueba  1.2750
red neuronal         entrenamiento  1.0882   prueba  1.1695


## Paso 4: el registro, ordenado

In [5]:
# Deja en corridas las corridas del experimento (mlflow.search_runs),
# ordenadas de menor a mayor mae_prueba, y en mejor_run_id el run_id de la
# primera.
corridas = (
    mlflow.search_runs(experiment_names=[EXPERIMENTO])
    .sort_values("metrics.mae_prueba")
    .reset_index(drop=True)
)

mejor_run_id = corridas.loc[0, "run_id"]

display(corridas[[
    "tags.mlflow.runName",
    "metrics.mae_entrenamiento",
    "metrics.mae_prueba",
    "metrics.segundos",
    "run_id",
]])

,tags.mlflow.runName,metrics.mae_entrenamiento,metrics.mae_prueba,metrics.segundos,run_id
0,red neuronal,1.088204,1.169505,1.395101,5d8afdb660214001bb6748089e51435c
1,red neuronal,1.088204,1.169505,1.304000,b4f89f0a89024ff1af774c804798a146
2,gradient boosting,1.070539,1.274971,0.151662,787fb6f4b999474393ca80f6599c44ad
3,gradient boosting,1.070539,1.274971,0.167901,e048ef0e91884b26b0ce1543345c7f39
4,bosque aleatorio,0.474358,1.299323,0.881063,51fa891268934d119a1e6ec00a1e2493
5,bosque aleatorio,0.474358,1.299323,0.887095,c261467c7f5e4ef2a45bc2486a207467
6,ridge,2.493893,2.556469,0.012177,4cbf3f793fed46afab5205ef1321e1a8
7,ridge,2.493893,2.556469,0.014686,99e5c21b213d4c68a26283304ae96583
8,linea base,13.199786,12.530824,0.024161,30a6d6e91bf4469983db3c0e7bfbd599
9,linea base,13.199786,12.530824,0.039755,bd95f9a124f84e5280f18226e7cda8ea


## Paso 5: marcar el ganador y reproducirlo

In [6]:
# Marca la corrida ganadora con la etiqueta "candidato" = "final"
# (mlflow.MlflowClient().set_tag). Despues carga su modelo desde el
# registro con mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo"),
# calcula su error absoluto medio sobre X_prueba y dejalo en
# mae_reproducido.
mlflow.MlflowClient().set_tag(
    mejor_run_id,
    "candidato",
    "final"
)

modelo_recuperado = mlflow.sklearn.load_model(
    "runs:/" + mejor_run_id + "/modelo"
)

mae_reproducido = mean_absolute_error(
    y_prueba,
    modelo_recuperado.predict(X_prueba)
)

print("MAE registrado:", corridas.loc[0, "metrics.mae_prueba"])
print("MAE reproducido:", mae_reproducido)

MAE registrado: 1.169505229635501
MAE reproducido: 1.169505229635501


## Comprobación

In [7]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
registradas = None
if experimento is not None:
    registradas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
nombres = set()
if registradas is not None and len(registradas) > 0:
    nombres = set(registradas["tags.mlflow.runName"])

comprobar(1, experimento is not None)
comprobar(2, registradas is not None and "params.huella_datos" in registradas and (registradas["params.huella_datos"] == huella_datos).all())
comprobar(3, {"linea base", "ridge", "bosque aleatorio", "gradient boosting", "red neuronal"} <= nombres and registradas["metrics.mae_prueba"].notna().all())
comprobar(4, corridas is not None and mejor_run_id == corridas.loc[0, "run_id"] and corridas.loc[0, "tags.mlflow.runName"] == "red neuronal")
etiqueta = None
if mejor_run_id is not None:
    etiqueta = mlflow.MlflowClient().get_run(mejor_run_id).data.tags.get("candidato")
comprobar(5, etiqueta == "final" and mae_reproducido is not None and abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9)

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK


## Si terminas antes

Abre la interfaz de MLflow (pestaña **Ports** de VS Code, puerto 5050, ícono del globo), entra al experimento `tarifa-taxis-comparacion`, selecciona las cinco corridas y compáralas con **Compare**. Además del error, ¿qué te dice el registro de la red neuronal que no te diría una tabla escrita a mano?